# D9 · Fundamentos de Machine Learning — Lección

**Bootcamp de Datos para Funcionarios Públicos · Formación Pública**
Línea B · *Data Scientist* · Semana 10 · Se ejecuta en Google Colab.\n\n> 🚀 **Cómo se trabaja:** lee, ejecuta cada celda con **▶︎** (o `Shift`+`Enter`) y completa los `TODO`. Cada ejercicio termina en una **celda de chequeo** que muestra ✅ si está bien o una pista si todavía no.

---

## ¿Qué vas a lograr?

En D8 aprendiste a **fabricar *features*** con SQL. Ahora usarás esas variables para lo que
de verdad persigue la ciencia de datos: **entrenar un modelo que aprenda un patrón a partir
de ejemplos y luego prediga casos nuevos**. Este es tu primer modelo de *machine learning*,
y lo importante no es el algoritmo (es sencillo), sino los **hábitos correctos**: separar un
conjunto de prueba, entrenar, evaluar con honestidad y entender el enemigo número uno, el
**sobreajuste**.

**Competencia de salida:** plantear un problema de aprendizaje supervisado (definir *features*
y objetivo), dividir los datos en entrenamiento y prueba, entrenar un modelo de regresión con
scikit-learn, evaluarlo de forma honesta en datos que no vio, y usarlo para predecir un caso nuevo.

**Dato real:** cantidad de artículos y monto total en compras públicas de alimentos de MercadoPúblico / ChileCompra.

**Entregable:** que las **4 celdas de chequeo** muestren ✅.


## Un puente desde D8 (y por qué cambiamos de dato)

En D8 construiste una *tabla analítica* de sismos: una fila por región. Para **entrenar** un
modelo, sin embargo, necesitamos **muchas filas con una señal clara**, y 6 regiones son pocas.
Así que cambiamos a una muestra representativa de compras públicas reales de alimentos. Lo que
aprendiste igual viaja: *features* → tabla → modelo. Hoy el objetivo a predecir será el
**monto total de la compra (CLP)**, y la *feature* será la **cantidad** de artículos comprada.


## 1. ¿Qué es el *machine learning*?

Programar "a la antigua" es **escribir reglas a mano**: *si la cantidad es mayor a 100, entonces
el monto es...*. El *machine learning* le da vuelta a esto: en vez de escribir la regla, le mostramos
al computador **muchos ejemplos** y dejamos que **descubra la regla solo**.

> 🧠 **Analogía.** No le enseñas a un niño a reconocer perros dándole una lista de reglas
> ("4 patas, cola, ladra"). Le muestras muchos perros y, solo, aprende el patrón. El ML hace
> eso con datos.

### Aprendizaje supervisado: el caso más común
Le damos al modelo ejemplos donde conocemos **la respuesta correcta**, y aprende a predecirla:

- **Features (`X`):** las variables de entrada, lo que el modelo usa para predecir. Aquí, la **cantidad** de artículos.
- **Objetivo o etiqueta (`y`):** la respuesta que queremos predecir. Aquí, el **monto_total** de la compra.
- El modelo aprende una función que, dado `X`, estima `y`.

Según qué sea `y`, hay dos sabores:

| Tipo | Qué predice | Ejemplo |
| --- | --- | --- |
| **Regresión** | un **número** | temperatura, monto, demanda |
| **Clasificación** | una **categoría** | ¿perceptible sí/no?, riesgo alto/medio/bajo |

Hoy haremos **regresión** (predecir un número). La clasificación llega en D11.

Ejecuta la celda de preparación: carga los datos reales y dibuja la relación que vamos a modelar.


In [ ]:
# ── Preparación del entorno (ejecuta esta celda primero) ──────────────────────
import os
import urllib.request
import pandas as pd
import matplotlib.pyplot as plt

# Si el archivo no existe localmente (ej: en Colab), intentamos descargarlo desde GitHub
if not os.path.exists("compras_ml.csv"):
    try:
        url = "https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/B2-fundamentos-de-ml/compras_ml.csv"  # Reemplazar por la URL real del repositorio al publicar
        urllib.request.urlretrieve(url, "compras_ml.csv")
        print("compras_ml.csv descargado automáticamente.")
    except Exception:
        print("Aviso: No se pudo descargar compras_ml.csv automáticamente. Si estás en Colab, asegúrate de subirlo manualmente.")

# Datos REALES de compras públicas del rubro alimentos
df = pd.read_csv("compras_ml.csv")

# Miremos la relacion ANTES de modelar: un grafico vale mas que mil palabras.
plt.figure(figsize=(7, 4.5))
plt.scatter(df["cantidad"], df["monto_total"], color="#c0392b", alpha=0.5)
plt.xlabel("Cantidad de artículos")
plt.ylabel("Monto total de la compra (CLP)")
plt.title("Relación entre cantidad de artículos y monto total")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"{len(df)} compras reales cargadas. Esta es nuestra tabla:")
df

## 2. La regla de oro: separar un conjunto de prueba

Aquí está la idea más importante de todo el módulo. Si entrenamos el modelo con **todos** los
datos y luego lo evaluamos con esos **mismos** datos, nos engañamos: el modelo puede salir
"perfecto" simplemente porque **memorizó las respuestas**, sin haber aprendido nada general.

> 🧠 **Analogía.** Es como rendir una prueba habiendo visto antes el solucionario. Sacas un 7,
> pero no demuestra que aprendiste. La verdadera evaluación es con preguntas que **no** habías visto.

Por eso partimos los datos en dos:
- **Entrenamiento (train):** los ejemplos con los que el modelo **aprende**.
- **Prueba (test):** ejemplos que el modelo **no ve** durante el entrenamiento y que usamos solo
  al final para medir qué tan bien **generaliza** a casos nuevos.

Cuando un modelo va muy bien en entrenamiento pero mal en prueba, decimos que hay
**sobreajuste** (*overfitting*): memorizó en vez de aprender el patrón. Reservar el conjunto de
prueba es lo que nos permite **detectarlo**.

> 🔗 **Conexión con D8.** Allá hablamos de *data leakage*: nunca dejar que información del futuro
> se filtre a una *feature*. Aquí es la misma ética: el conjunto de prueba **no puede tocarse**
> durante el entrenamiento, o estaríamos haciendo trampa con nosotros mismos.

`train_test_split` hace la división al azar por nosotros. Fijamos `random_state=42` para que el
sorteo sea **siempre el mismo** (resultados reproducibles).


### ✍️ Ejercicio 1 — Separar `X`, `y` y reservar la prueba
Define `X` (la *feature* `cantidad`, con **doble corchete** para que sea una tabla) e `y`
(la columna `monto_total`), y divídelos con `train_test_split` usando `test_size=0.3` y
`random_state=42`. Completa la celda.


In [ ]:
# X = "features" (lo que el modelo usa para predecir). Doble corchete -> tabla.
# y = "objetivo" o "etiqueta" (lo que queremos predecir). Un corchete -> columna.
X = df[["cantidad"]]
y = df["monto_total"]

from sklearn.model_selection import train_test_split
# TODO: divide X e y en entrenamiento y prueba con train_test_split,
#       usando test_size=0.3 y random_state=42. Reemplaza la linea de abajo.
X_train, X_test, y_train, y_test = X, X, y, y   # <-- PLACEHOLDER: aun no divide nada

print("Entrenamiento:", len(X_train), "compras | Prueba:", len(X_test), "compras")


In [ ]:
# ── Celda de chequeo · Ejercicio 1 ───────────────────────────────────────────
try:
    assert X_train.shape[1] == 1 and list(X_train.columns) == ["cantidad"], \
        "X debe tener exactamente la columna 'cantidad' (usa doble corchete: df[['cantidad']])."
    assert len(X_train) == 5188 and len(X_test) == 2224, \
        f"Con test_size=0.3 deben quedar 5188 de entrenamiento y 2224 de prueba; tienes {len(X_train)} y {len(X_test)}."
    assert len(y_train) == 5188 and len(y_test) == 2224, "y_train/y_test no calzan con la division."
    print("✅ Correcto: separaste features (X) y objetivo (y), y reservaste un conjunto de prueba.")
except Exception as e:
    print("❌ Aun no. Revisa tu division de datos y vuelve a intentar.")
    print("   Pista:", e)


## 3. Entrenar tu primer modelo

Usaremos **regresión lineal**: el modelo más sencillo y transparente. Busca la **mejor línea
recta** que relaciona la *feature* con el objetivo; aquí, cuántos pesos (CLP) aumenta el monto
total por cada unidad adicional comprada. En scikit-learn, todos los modelos se usan igual, con dos
pasos:

```python
modelo = LinearRegression()
modelo.fit(X_train, y_train)     # 1) APRENDER de los ejemplos de entrenamiento
y_pred = modelo.predict(X_test)  # 2) PREDECIR sobre datos nuevos (los de prueba)
```

`fit` is "estudiar"; `predict` is "responder". Este patrón `fit` / `predict` es **idéntico**
para casi todos los modelos de scikit-learn (árboles, etc.), así que lo que aprendes aquí te
servirá en D10 y D11 sin cambios.

> ⚠️ **Errores típicos**
> - **Entrenar con los datos de prueba.** Siempre `fit` con `X_train`, nunca con `X_test`.
> - **Pasar `y` con doble corchete.** El objetivo va con **un** corchete (`df["monto_total"]`);
>   las *features* van con doble corchete (`df[["cantidad"]]`).
> - **Olvidar entrenar antes de predecir.** Si llamas a `predict` sin `fit`, el modelo "no sabe nada"
>   todavía y dará error.


### ✍️ Ejercicio 2 — Crear, entrenar y predecir
Crea un `LinearRegression`, entrénalo con `X_train`/`y_train` (`.fit`) y predice sobre `X_test`
(`.predict`), guardando el resultado en `y_pred`. Completa la celda.


In [ ]:
from sklearn.linear_model import LinearRegression

modelo = LinearRegression()
# TODO: 1) entrena el modelo con los datos de ENTRENAMIENTO -> modelo.fit(X_train, y_train)
#       2) predice sobre X_test y guarda el resultado -> y_pred = modelo.predict(X_test)
y_pred = [0.0] * len(X_test)   # <-- PLACEHOLDER: reemplaza por la prediccion real

print("Recuerda entrenar (.fit) y luego predecir (.predict).")


In [ ]:
# ── Celda de chequeo · Ejercicio 2 ───────────────────────────────────────────
import numpy as np
try:
    assert hasattr(modelo, "coef_"), "Aun no entrenaste el modelo: falta modelo.fit(X_train, y_train)."
    esperado = modelo.predict(X_test)
    assert len(y_pred) == len(esperado), "y_pred debe tener una prediccion por cada compra de X_test."
    assert np.allclose(np.array(y_pred, dtype=float), esperado, atol=1e-6), \
        "y_pred deberia venir de modelo.predict(X_test)."
    print(f"✅ Correcto: entrenaste tu primer modelo. Aprendio que cada unidad adicional cuesta ~{modelo.coef_[0]:.2f} CLP.")
except Exception as e:
    print("❌ Aun no. Revisa tu entrenamiento y prediccion, y vuelve a intentar.")
    print("   Pista:", e)


## 4. Evaluar con honestidad

Un modelo sin evaluación es un acto de fe. Para regresión, una métrica clara es el **Error
Absoluto Medio (MAE)**: en promedio, **¿por cuántos grados se equivoca** el modelo? Mientras más
bajo, mejor. Lo crucial es calcularlo **sobre el conjunto de prueba**, porque mide el desempeño
en casos que el modelo no vio.

```python
from sklearn.metrics import mean_absolute_error
mae = mean_absolute_error(y_test, y_pred)
```

La celda de chequeo, además de validar tu MAE, te mostrará el error en **entrenamiento** y en
**prueba**. Si son parecidos, el modelo **generaliza** bien (no sobreajustó). Si el de
entrenamiento fuera mucho menor que el de prueba, sería la alarma clásica de **sobreajuste**.

> ⚠️ **Error típico:** reportar el error medido en los **datos de entrenamiento** como si fuera
> el desempeño real. Casi siempre es demasiado optimista. La verdad está en la prueba.


## 4. Evaluar con honestidad

Un modelo sin evaluación es un acto de fe. Para regresión, una métrica clara es el **Error
Absoluto Medio (MAE)**: en promedio, **¿por cuántos pesos (CLP) se equivoca** el modelo? Mientras más
bajo, mejor. Lo crucial es calcularlo **sobre el conjunto de prueba**, porque mide el desempeño
en casos que el modelo no vio.

```python
from sklearn.metrics import mean_absolute_error
mae = mean_absolute_error(y_test, y_pred)
```

La celda de chequeo, además de validar tu MAE, te mostrará el error en **entrenamiento** y en
**prueba**. Si son parecidos, el modelo **generaliza** bien (no sobreajustó). Si el de
entrenamiento fuera mucho menor que el de prueba, sería la alarma clásica de **sobreajuste**.

> ⚠️ **Error típico:** reportar el error medido en los **datos de entrenamiento** como si fuera
> el desempeño real. Casi siempre es demasiado optimista. La verdad está en la prueba.


In [ ]:
from sklearn.metrics import mean_absolute_error

# TODO: calcula el error absoluto medio entre y_test (lo real) y y_pred (lo estimado)
#       usando mean_absolute_error(y_test, y_pred). Guardalo en mae.
mae = 0.0   # <-- PLACEHOLDER

print(f"Error medio en prueba: {mae:.2f} CLP")


In [ ]:
# ── Celda de chequeo · Ejercicio 3 ───────────────────────────────────────────
from sklearn.metrics import mean_absolute_error
try:
    assert hasattr(modelo, "coef_"), "Primero completa el ejercicio 2 (entrenar el modelo con .fit)."
    esperado = modelo.predict(X_test)
    mae_real = mean_absolute_error(y_test, esperado)
    assert abs(float(mae) - mae_real) < 0.05, f"Tu MAE deberia ser ~{mae_real:.2f} CLP."
    mae_train = mean_absolute_error(y_train, modelo.predict(X_train))
    print(f"✅ Correcto. Error en prueba ~{mae_real:.2f} CLP; en entrenamiento ~{mae_train:.2f} CLP.")
    print("   Que sean parecidos es buena senal: el modelo GENERALIZA, no memorizo.")
except Exception as e:
    print("❌ Aun no. Revisa tu calculo del MAE y vuelve a intentar.")
    print("   Pista:", e)


## 5. Usar el modelo: predecir un caso nuevo

Para esto entrenamos modelos: **estimar lo que no hemos medido**. Una vez entrenado, podemos
darle una cantidad cualquiera y pedirle el monto total esperado. scikit-learn pide la entrada con
la **misma forma** que usamos para entrenar (una tabla con la columna `cantidad`):

```python
nueva_compra = pd.DataFrame({"cantidad": [100]})
monto_estimado = modelo.predict(nueva_compra)[0]
```

> 🧠 **Sentido público.** Cambia "cantidad → monto_total" por "características de un trámite →
> tiempo de demora", o "datos de una comuna → demanda de un programa", y tienes la forma de
> innumerables modelos útiles para el Estado. La mecánica es la misma que acabas de aprender.


### ✍️ Ejercicio 4 — Predecir una cantidad nueva
Estima la temperatura media para una compra de **100 artículos**. Arma un
`DataFrame` con esa cantidad, pásalo a `modelo.predict(...)` y toma el primer valor con `[0]`.
Guárdalo en `temp_estimada` (conservamos este nombre de variable para no romper chequeos legacy). Completa la celda.


In [ ]:
import pandas as pd

# TODO: predice el monto total de una compra de 100 articulos.
#       Pista: arma un DataFrame con esa cantidad y pasalo a modelo.predict(...);
#       toma el primer (y unico) valor con [0]. Guardalo en temp_estimada.
#   nueva_compra = pd.DataFrame({"cantidad": [100]})
#   temp_estimada = modelo.predict(nueva_compra)[0]
temp_estimada = 0.0   # <-- PLACEHOLDER

print(f"Monto total estimado para 100 articulos: {temp_estimada:.1f} CLP")


In [ ]:
# ── Celda de chequeo · Ejercicio 4 ───────────────────────────────────────────
import pandas as pd
try:
    assert hasattr(modelo, "coef_"), "Primero completa los ejercicios 1 y 2 (entrenar el modelo)."
    esperado = float(modelo.predict(pd.DataFrame({"cantidad": [100]}))[0])
    assert abs(float(temp_estimada) - esperado) < 0.1, \
        f"Para 100 articulos el modelo deberia estimar ~{esperado:.1f} CLP."
    print(f"✅ Correcto: usaste el modelo para predecir un caso nuevo (~{esperado:.1f} CLP). 🎯")
    print("   Eso es, en el fondo, para lo que sirve un modelo: estimar lo que no hemos medido.")
except Exception as e:
    print("❌ Aun no. Revisa tu prediccion de la cantidad nueva y vuelve a intentar.")
    print("   Pista:", e)


## 6. Cierre

Entrenaste tu primer modelo de machine learning y, más importante, lo hiciste **bien**:

1. Planteaste un problema **supervisado**: *features* (`X`) → objetivo (`y`), y distinguiste
   **regresión** (predecir un número) de **clasificación**.
2. Aplicaste la **regla de oro**: separar un conjunto de **prueba** para poder detectar el
   **sobreajuste** y medir la **generalización**.
3. Usaste el patrón universal de scikit-learn: **`fit`** (aprender) y **`predict`** (responder).
4. Evaluaste con **honestidad** (MAE en prueba) y usaste el modelo para **predecir un caso nuevo**.

Estas ideas son la base de todo lo que viene. En **D10 · Modelos basados en árboles** cambiaremos
la regresión lineal por modelos más potentes, pero con el **mismo** `fit`/`predict` y la **misma**
disciplina de entrenamiento y prueba.

### Mini-glosario
- **Aprendizaje supervisado:** aprender a partir de ejemplos con la respuesta conocida.
- **Features (`X`) / objetivo (`y`):** entradas del modelo / lo que se quiere predecir.
- **Regresión / clasificación:** predecir un número / predecir una categoría.
- **Entrenamiento y prueba:** datos para aprender / datos reservados para evaluar.
- **Sobreajuste (*overfitting*):** el modelo memoriza en vez de aprender; va bien en
  entrenamiento y mal en datos nuevos.
- **`fit` / `predict`:** entrenar el modelo / pedirle una predicción.
- **MAE:** error absoluto medio; en promedio, cuánto se equivoca el modelo.

---
*Fuente de datos: Dirección de Compras y Contratación Pública (ChileCompra / MercadoPúblico).*
*Contenido bajo licencia CC BY 4.0 · Formación Pública.*
